<a href="https://colab.research.google.com/github/daliaray/Daliaray/blob/main/PREDICTWEEK.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [25]:
import pandas as pd
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

# Load data
data = pd.read_csv('/content/attendance_data_improved.csv')

sequence_length = 4

# Encode target
label_encoder = LabelEncoder()
data['semaine_encoded'] = label_encoder.fit_transform(data['semaine'])

# Encode input
status_map = {'present': 1, 'absent': 0}
data['status_numeric'] = data['status'].map(status_map)

# Prepare sequences
X, y = [], []

for student_id, student_data in data.groupby('student_id'):
    student_data = student_data.sort_values('date')
    if len(student_data) > sequence_length:
        for i in range(len(student_data) - sequence_length):
            X.append(student_data['status_numeric'].values[i:i+sequence_length])
            y.append(student_data['semaine_encoded'].values[i+sequence_length])

X = np.array(X)
y = np.array(y)

# Reshape for LSTM
X = X.reshape((X.shape[0], X.shape[1], 1))

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.5, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# Build model (correct version)
model = Sequential([
    Input(shape=(sequence_length, 1)),
    LSTM(32),
    Dense(3, activation='softmax')
])

model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# 🔥 Early Stopping
early_stop = EarlyStopping(
    monitor='val_loss',        # نراقب validation
    patience=3,               # نسمح 3 epochs بدون تحسن
    restore_best_weights=True # يرجع لأفضل model
)

# Train
model.fit(
    X_train, y_train,
    epochs=50,   # نخليها كبيرة وهو يوقف وحدو
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=[early_stop]
)

# Evaluate
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f'Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}')

# Predict
future_sequence = np.array([[0, 1, 0, 1]])
future_sequence = future_sequence.reshape((1, sequence_length, 1))

predictions = model.predict(future_sequence)
predicted_label = np.argmax(predictions, axis=1)
predicted_category = label_encoder.inverse_transform(predicted_label)

print(f'Predicted Category: {predicted_category[0]}')

Epoch 1/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.6188 - loss: 0.9044 - val_accuracy: 0.6369 - val_loss: 0.7798
Epoch 2/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.6604 - loss: 0.7325 - val_accuracy: 0.6369 - val_loss: 0.7347
Epoch 3/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.6623 - loss: 0.6909 - val_accuracy: 0.6715 - val_loss: 0.6914
Epoch 4/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.7081 - loss: 0.6485 - val_accuracy: 0.7477 - val_loss: 0.6505
Epoch 5/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7365 - loss: 0.6182 - val_accuracy: 0.7477 - val_loss: 0.6227
Epoch 6/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7385 - loss: 0.5989 - val_accuracy: 0.7031 - val_loss: 0.6184
Epoch 7/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7415 - loss: 0.5936 - val_accuracy: 0.7554 - val_loss: 0.5984
Epoch 8/50
82/82 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.7477 - loss: 0.5856 - val_accuracy: 0.7708 - val_loss:

In [22]:
import pandas as pd
import numpy as np

# PARAMETERS
num_students = 200
num_weeks = 30
sequence_length = 4

data = []

for student_id in range(1, num_students + 1):

    # بداية عشوائية لمستوى الطالب
    base_attendance = np.random.uniform(0.5, 0.95)

    history = []

    for week in range(num_weeks):

        # simulate attendance
        status = np.random.choice(
            ['present', 'absent'],
            p=[base_attendance, 1 - base_attendance]
        )

        status_num = 1 if status == 'present' else 0
        history.append(status_num)

        # 🔥 label يعتمد على آخر 4 أسابيع
        if len(history) >= sequence_length:
            avg = np.mean(history[-sequence_length:])

            if avg >= 0.75:
                semaine = 'green'
            elif avg >= 0.5:
                semaine = 'normal'
            else:
                semaine = 'red'
        else:
            semaine = 'normal'  # البداية فقط

        # optional: trend (تحسن أو تدهور)
        trend = np.random.uniform(-0.02, 0.02)
        base_attendance = min(max(base_attendance + trend, 0.3), 1.0)

        data.append([
            student_id,
            f"2023-W{week}",
            status,
            semaine
        ])

# CREATE DATAFRAME
df = pd.DataFrame(data, columns=[
    'student_id',
    'date',
    'status',
    'semaine'
])

# SAVE FILE
df.to_csv('/content/attendance_data_improved.csv', index=False)

print("Improved dataset created successfully!")
print(df.head())

Improved dataset created successfully!
   student_id     date   status semaine
0           1  2023-W0  present  normal
1           1  2023-W1  present  normal
2           1  2023-W2  present  normal
3           1  2023-W3  present   green
4           1  2023-W4  present   green
